# Paper-level stacked metrics (all papers)

**11 x-axis groups:** 10 papers + **Overall mean** (no separate overall-score group).

Per paper and method: stacked **TP similarity**, **factual rate**, **retrieval rate**.

**Overall mean** (rightmost): mean of those three metrics across all papers, per method.

Configure `provider_model_tag` and `n_fields_tag` in the setup cell.


In [1]:
from __future__ import annotations

import csv
import re
import sys
from pathlib import Path

try:
    import pandas as pd  # type: ignore
except Exception:  # pragma: no cover
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    p = start or Path.cwd()
    for _ in range(6):
        if (p / "outputs").exists() and (p / "src").exists():
            return p
        p = p.parent
    return start or Path.cwd()


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
outputs_root = repo_root / "outputs"
gt_path = repo_root / "data" / "wopke_paper_code" / "Database for combined sample 2015-03-05.csv"

# --- Configure which run outputs to load ---
provider_model_tag = "google_gemini-3-1-flash-lite-preview"
n_fields_tag = "42fields"

METHODS: dict[str, str] = {
    "direct_llm": "direct_llm",
    "workflow": "static_workflow",
    "MAS": "mas",
}

print("repo_root:", repo_root)
print("outputs_root:", outputs_root)
print("gt_path:", gt_path)
print("provider_model_tag:", provider_model_tag)
print("n_fields_tag:", n_fields_tag)
print("methods:", METHODS)


repo_root: /home/com3dian/Github/meta_analysis_agents
outputs_root: /home/com3dian/Github/meta_analysis_agents/outputs
gt_path: /home/com3dian/Github/meta_analysis_agents/data/wopke_paper_code/Database for combined sample 2015-03-05.csv
provider_model_tag: google_gemini-3-1-flash-lite-preview
n_fields_tag: 42fields
methods: {'direct_llm': 'direct_llm', 'workflow': 'static_workflow', 'MAS': 'mas'}


In [ ]:
def is_present(v) -> bool:
    if v is None:
        return False
    s = str(v).strip()
    if not s:
        return False
    s_low = s.lower()
    return s_low not in {"nan", "none", "null", "n/a", "na"}


def read_prediction_csv(csv_path: str) -> tuple[list[dict], list[str]]:
    with open(csv_path, "r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames or [])
        rows = [dict(r) for r in reader]
    return rows, fieldnames


def scan_latest_for_method(method_prefix: str) -> dict[int, str]:
    """Return {paper_id: csv_path} for latest run per paper_id."""
    if not outputs_root.exists():
        raise FileNotFoundError(f"outputs folder not found: {outputs_root}")

    needle = f"{method_prefix}_{provider_model_tag}_{n_fields_tag}_"
    paper_id_re = re.compile(r"^(?P<paper_id>\d+)_")
    date_re = re.compile(r"_(?P<date>\d{4}-\d{2}-\d{2})\.csv$")

    latest_by_pid: dict[int, dict] = {}
    for dated_dir in sorted(outputs_root.iterdir()):
        if not dated_dir.is_dir():
            continue
        if not re.match(r"^\d{4}-\d{2}-\d{2}$", dated_dir.name):
            continue

        for csv_path in dated_dir.glob("*.csv"):
            name = csv_path.name
            if needle not in name:
                continue

            m_pid = paper_id_re.match(name)
            m_date = date_re.search(name)
            if not m_pid or not m_date:
                continue

            pid = int(m_pid.group("paper_id"))
            date = m_date.group("date")

            prev = latest_by_pid.get(pid)
            if prev is None or date > prev["date"]:
                latest_by_pid[pid] = {"date": date, "csv_path": str(csv_path)}

    return {pid: v["csv_path"] for pid, v in latest_by_pid.items()}


def load_ground_truth_by_study_id() -> dict[int, list[dict]]:
    """Load combined-sample GT grouped by Study#, with standard field aliases."""
    from src.experimentutils.eval_utils import load_ground_truth_by_study_id as _load_gt
    if not gt_path.exists():
        raise FileNotFoundError(f"Ground truth CSV not found: {gt_path}")
    return _load_gt(str(gt_path))


# --- Load data ---
pred_by_method: dict[str, dict[int, str]] = {}
for method_label, method_prefix in METHODS.items():
    pred_by_method[method_label] = scan_latest_for_method(method_prefix)
    print(method_label, "papers:", len(pred_by_method[method_label]))

from src.experimentutils.eval_utils import wopke_100_shared_fields

gt_by_id = load_ground_truth_by_study_id()
print("GT papers:", len(gt_by_id))

some_pid = next(iter(gt_by_id.keys()))
some_gt_rows = gt_by_id.get(some_pid, [])
gt_keys = set(some_gt_rows[0].keys()) if some_gt_rows else set()
shared_fields = wopke_100_shared_fields(gt_keys)

print("shared_fields (standard ∩ GT):", len(shared_fields))
print("example shared_fields:", shared_fields[:10])
print("has Replications SC1:", "Replications SC1" in shared_fields)
print("has N Unit:", "N Unit" in shared_fields)


In [ ]:
from src.experimentutils.eval_utils import (
    infer_gt_spreadsheet_columns,
    match_records_greedy_with_units,
    overall_metrics_with_units,
)


def _gt_spreadsheet_for_fields(fields: list[str]) -> bool:
    return infer_gt_spreadsheet_columns(fields)


def overall_metrics(pred_rows: list[dict], gt_rows: list[dict], fields: list[str]) -> dict:
    return overall_metrics_with_units(
        pred_rows, gt_rows, fields, gt_spreadsheet=_gt_spreadsheet_for_fields(fields)
    )


def paper_label(pid: int) -> str:
    rows = gt_by_id.get(pid, [])
    if not rows:
        return str(pid)
    rec = rows[0]
    author = rec.get("Author")
    title = rec.get("Title")
    a = author.strip() if isinstance(author, str) else ""
    t = title.strip() if isinstance(title, str) else ""
    if a and t:
        return f"{a} — {t}"
    return a or t or str(pid)


def first_author_full_name(author: str | None) -> str:
    """First author from GT Author (semicolon-separated list)."""
    if not author or not isinstance(author, str):
        return ""
    s = author.strip()
    if not s:
        return ""
    return s.split(";")[0].strip()


def paper_publication_year(pid: int) -> str:
    rows = gt_by_id.get(pid, [])
    if not rows:
        return ""
    val = rows[0].get("Year of publication")
    if val is None:
        return ""
    try:
        return str(int(float(val)))
    except (TypeError, ValueError):
        s = str(val).strip()
        return s if s else ""


def paper_index_label(pid: int) -> str:
    """X-axis: first author + publication year."""
    rows = gt_by_id.get(pid, [])
    if not rows:
        return str(pid)
    author = first_author_full_name(rows[0].get("Author"))
    year = paper_publication_year(pid)
    if author and year:
        return f"{author}\n{year}"
    return author or year or str(pid)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

METHOD_COLORS = {"direct_llm": "#4C72B0", "workflow": "#55A868", "MAS": "#C44E52"}

STACK_METRIC_KEYS = ("similarity", "factual_rate", "retrieval_rate")
OVERALL_SCORE_KEY = "overall_score"
METRIC_KEYS = STACK_METRIC_KEYS + (OVERALL_SCORE_KEY,)

METRIC_LABELS = {
    "similarity": "TP similarity",
    "factual_rate": "Factual rate (TP/(TP+FP))",
    "retrieval_rate": "Retrieval rate (TP/(TP+FN))",
    OVERALL_SCORE_KEY: "Overall score (true precision)",
}
METRIC_HATCHES = {
    "similarity": "",
    "factual_rate": "///",
    "retrieval_rate": "...",
    OVERALL_SCORE_KEY: "xx",
}


def _safe01(v: float) -> float:
    return float(v) if v == v else 0.0


def paper_three_metrics(
    paper_id: int,
    method: str,
    *,
    fields: list[str] | None = None,
) -> dict[str, float]:
    """Unit-aware paper scores: 3 stack components + overall score."""
    gt_rows = gt_by_id.get(paper_id, [])
    if not gt_rows:
        raise KeyError(f"paper_id={paper_id} not found in GT")

    use_fields = fields or shared_fields
    csv_path = pred_by_method.get(method, {}).get(paper_id)
    pred_rows = []
    if csv_path is not None:
        pred_rows, _ = read_prediction_csv(csv_path)

    met = overall_metrics(pred_rows, gt_rows, use_fields)
    return {
        "similarity": _safe01(met.get("tp_avg_similarity", float("nan"))),
        "factual_rate": _safe01(met.get("precision", float("nan"))),
        "retrieval_rate": _safe01(met.get("recall", float("nan"))),
        OVERALL_SCORE_KEY: _safe01(met.get("true_precision", float("nan"))),
    }


def all_papers_metrics_df(
    paper_ids: list[int] | None = None,
    *,
    methods: list[str] | None = None,
    fields: list[str] | None = None,
) -> "pd.DataFrame":
    use_methods = methods or ["direct_llm", "workflow", "MAS"]
    pids = paper_ids or sorted(
        set().union(*(set(pred_by_method[m].keys()) for m in METHODS.keys()))
    )

    rows: list[dict] = []
    for pid in pids:
        for m in use_methods:
            scores = paper_three_metrics(pid, m, fields=fields)
            rows.append(
                {
                    "paper_id": pid,
                    "method": m,
                    **scores,
                    "paper": paper_index_label(pid),
                }
            )
    return pd.DataFrame(rows)


def overall_mean_metrics_df(
    paper_ids: list[int] | None = None,
    *,
    methods: list[str] | None = None,
    fields: list[str] | None = None,
) -> "pd.DataFrame":
    """Per-method means across papers (stack components + overall score)."""
    df = all_papers_metrics_df(paper_ids, methods=methods, fields=fields)
    use_methods = methods or ["direct_llm", "workflow", "MAS"]
    rows = []
    for m in use_methods:
        sub = df[df["method"] == m]
        row = {"method": m}
        for k in STACK_METRIC_KEYS:
            row[k] = float(sub[k].mean())
        row[OVERALL_SCORE_KEY] = float(sub[OVERALL_SCORE_KEY].mean())
        row["stacked_total"] = sum(row[k] for k in STACK_METRIC_KEYS)
        rows.append(row)
    return pd.DataFrame(rows)


def plot_paper_metrics_stacked_bars(
    paper_ids: list[int] | None = None,
    *,
    methods: list[str] | None = None,
    fields: list[str] | None = None,
    figsize: tuple[float, float] | None = None,
    bar_width: float = 0.2,
):
    """Per paper: 3-method stacked bars. One extra group: overall mean across papers."""
    use_methods = methods or ["direct_llm", "workflow", "MAS"]
    df = all_papers_metrics_df(paper_ids, methods=use_methods, fields=fields)
    pids = sorted(df["paper_id"].unique())
    mean_df = overall_mean_metrics_df(paper_ids, methods=use_methods, fields=fields)

    n_methods = len(use_methods)
    n_papers = len(pids)
    # x positions: papers + overall mean (stacked)
    overall_mean_x = n_papers
    n_groups = n_papers + 1
    x = np.arange(n_groups)

    offsets = (np.arange(n_methods) - (n_methods - 1) / 2) * bar_width

    if figsize is None:
        figsize = (max(14, n_groups * 1.35), 6.5)

    fig, ax = plt.subplots(figsize=figsize)

    def _draw_stack(x_pos: float, heights: dict[str, float], color: str, *, alpha: float = 1.0):
        bottom = 0.0
        for key in STACK_METRIC_KEYS:
            ax.bar(
                x_pos,
                heights[key],
                bar_width,
                bottom=bottom,
                color=color,
                edgecolor="black",
                linewidth=0.6,
                hatch=METRIC_HATCHES[key],
                alpha=alpha,
            )
            bottom += heights[key]

    for mi, m in enumerate(use_methods):
        sub = df[df["method"] == m].set_index("paper_id").loc[pids]
        color = METHOD_COLORS.get(m, "steelblue")

        for pi, pid in enumerate(pids):
            stack_h = {k: float(sub.loc[pid, k]) for k in STACK_METRIC_KEYS}
            _draw_stack(x[pi] + offsets[mi], stack_h, color)

        mean_row = mean_df[mean_df["method"] == m].iloc[0]
        stack_mean = {k: float(mean_row[k]) for k in STACK_METRIC_KEYS}
        _draw_stack(overall_mean_x + offsets[mi], stack_mean, color, alpha=0.55)

    short_labels = [paper_index_label(pid) for pid in pids] + ["Overall\nmean"]

    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, rotation=0, ha="center", fontsize=8)
    ax.axvline(overall_mean_x - 0.5, color="gray", linestyle="--", linewidth=1.0, alpha=0.6)

    ax.set_ylabel("Score (each segment / overall bar ∈ [0, 1])")
    ax.set_ylim(0, 3.25)
    ax.set_title(
        "Paper metrics by method (+ overall mean across papers)",
        fontsize=11,
    )
    for y in (1.0, 2.0, 3.0):
        ax.axhline(y, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)
    ax.grid(axis="y", linestyle=":", alpha=0.35)

    stack_handles = [
        Patch(facecolor="white", edgecolor="black", hatch=METRIC_HATCHES[k], label=METRIC_LABELS[k])
        for k in STACK_METRIC_KEYS
    ]
    method_handles = [
        Patch(facecolor=METHOD_COLORS.get(m, "steelblue"), edgecolor="black", label=m)
        for m in use_methods
    ]
    leg1 = ax.legend(handles=stack_handles, title="Metric", loc="upper left")
    ax.add_artist(leg1)
    ax.legend(handles=method_handles, title="Method", loc="upper right")

    plt.tight_layout()
    plt.show()
    return df, mean_df


paper_ids = sorted(set().union(*(set(pred_by_method[m].keys()) for m in METHODS.keys())))
print("paper_ids:", paper_ids)
print("X-axis labels:", [paper_index_label(pid) for pid in paper_ids])

metrics_df, mean_df = plot_paper_metrics_stacked_bars(paper_ids)
print("\nOverall mean scores (per method, across papers):")
print(mean_df.to_string(index=False))
mean_df


## Per-field average across papers

Grouped bars: **x-axis = field**, **3 bars per field** = `direct_llm`, `workflow`, `MAS`.

Heights = mean unit-aware field score, averaged over greedy-matched GT rows **per paper**, then **across all papers**.
Rightmost group **Overall mean** = mean across all fields (per method).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

FIELD_METHOD_COLORS = {"direct_llm": "#4C72B0", "workflow": "#55A868", "MAS": "#C44E52"}


def per_field_avg_across_papers_df(
    paper_ids: list[int] | None = None,
    *,
    fields: list[str] | None = None,
    methods: list[str] | None = None,
) -> "pd.DataFrame":
    """Mean field score per method, averaged across papers (unit-aware greedy match)."""
    use_fields = fields or shared_fields
    use_methods = methods or ["direct_llm", "workflow", "MAS"]
    pids = paper_ids or sorted(
        set().union(*(set(pred_by_method[m].keys()) for m in METHODS.keys()))
    )
    gt_ss = _gt_spreadsheet_for_fields(use_fields)

    rows: list[dict] = []
    for f in use_fields:
        row: dict = {"field": f}
        for m in use_methods:
            paper_avgs: list[float] = []
            for pid in pids:
                gt_rows = gt_by_id.get(pid, [])
                if not gt_rows:
                    continue
                csv_path = pred_by_method.get(m, {}).get(pid)
                if csv_path is None:
                    continue
                pred_rows, _ = read_prediction_csv(csv_path)
                matches = match_records_greedy_with_units(
                    pred_rows, gt_rows, use_fields, gt_spreadsheet=gt_ss
                )
                scores = [ev.field_scores.get(f, 0.0) for _, _, ev in matches]
                if scores:
                    paper_avgs.append(float(np.mean(scores)))
            row[m] = float(np.mean(paper_avgs)) if paper_avgs else float("nan")
        rows.append(row)

    return pd.DataFrame(rows)


def plot_per_field_avg_across_papers(
    paper_ids: list[int] | None = None,
    *,
    fields: list[str] | None = None,
    methods: list[str] | None = None,
    figsize: tuple[float, float] | None = None,
    include_overall_mean_column: bool = True,
):
    use_methods = methods or ["direct_llm", "workflow", "MAS"]
    df = per_field_avg_across_papers_df(paper_ids, fields=fields, methods=use_methods)

    if include_overall_mean_column:
        overall_row = {"field": "Overall mean"}
        for m in use_methods:
            overall_row[m] = float(df[m].mean())
        df = pd.concat([df, pd.DataFrame([overall_row])], ignore_index=True)

    n_groups = len(df)
    if figsize is None:
        figsize = (max(14, n_groups * 0.38), 6.5)

    x = np.arange(n_groups)
    width = 0.8 / len(use_methods)

    fig, ax = plt.subplots(figsize=figsize)
    for i, m in enumerate(use_methods):
        offset = (i - (len(use_methods) - 1) / 2) * width
        ax.bar(
            x + offset,
            df[m].values,
            width=width,
            label=m,
            color=FIELD_METHOD_COLORS.get(m, "steelblue"),
            alpha=0.9,
        )

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Mean field score (avg across papers)")
    ax.set_xlabel("Field")
    ax.set_title("Per-field scores by method (mean across papers)", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(df["field"], rotation=90, ha="center", fontsize=7)
    if include_overall_mean_column and n_groups > 1:
        ax.axvline(n_groups - 1.5, color="gray", linestyle="--", linewidth=1.0, alpha=0.6)
    ax.legend(title="Method", loc="upper right")
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)
    ax.grid(axis="y", linestyle=":", alpha=0.35)
    plt.tight_layout()
    plt.show()
    return df


field_df = plot_per_field_avg_across_papers(paper_ids)
field_df
